# Initialize `oil_network_data_loader` schema — orchestrator

Thin orchestrator that runs every per-source loader in sequence. Each loader notebook owns its own subset of the `oil_network_data_loader` schema (raw staging + per-source catalogue + per-source facts) and is independently runnable for development.

## Per-source loaders

| Loader | Source | Status |
|---|---|---|
| [load_eia.ipynb](load_eia.ipynb) | EIA Open Data API v2 (27 datasets across `petroleum/*` + `steo`) | active |
| `load_cer.ipynb` | Canada Energy Regulator — per-pipeline cross-border flows | future |
| `load_aer.ipynb` | Alberta Energy Regulator (AER ST3) — Canadian production by play | future |

Adding a new source is a 2-step change: (1) drop a new `load_<source>.ipynb` next to this orchestrator; (2) append its filename to the `LOADERS` list below.

In [1]:
import subprocess
import sys
from datetime import datetime
from pathlib import Path

LOADERS = [
    "load_eia.ipynb",
    # "load_cer.ipynb",     # future
    # "load_aer.ipynb",     # future
]

TIMEOUT_SEC = 1800


def run_notebook(nb_path):
    cmd = [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--inplace",
        f"--ExecutePreprocessor.timeout={TIMEOUT_SEC}",
        nb_path,
    ]
    return subprocess.run(cmd, capture_output=True, text=True)


failures = []
for nb in LOADERS:
    if not Path(nb).exists():
        print(f"  ✗ {nb}: file missing")
        failures.append((nb, "file missing"))
        continue
    print(f"[{datetime.now():%H:%M:%S}] ▶ Running {nb}...")
    res = run_notebook(nb)
    if res.returncode != 0:
        # nbconvert dumps tracebacks into stderr
        tail = res.stderr.strip().splitlines()[-20:]
        failures.append((nb, "\n".join(tail)))
        print(f"[{datetime.now():%H:%M:%S}]   ✗ FAILED (exit {res.returncode}) — see summary at end")
    else:
        print(f"[{datetime.now():%H:%M:%S}]   ✓ done")

print()
if failures:
    print(f"❌ {len(failures)} loader(s) failed:\n")
    for nb, err in failures:
        print(f"--- {nb} ---")
        print(err)
        print()
else:
    print(f"✅ all {len(LOADERS)} loader(s) ran cleanly")

[17:17:51] ▶ Running load_eia.ipynb...


[17:20:16]   ✓ done

✅ all 1 loader(s) ran cleanly
